# ATELIER 3 : COUCHE GOLD — TABLES ANALYTIQUES ET KPIs

**Lakehouse** : `Lakehouse_gold` · **Notebook** : `NB_03_Gold`
**Objectif** : construire les **6 tables analytiques** prêtes pour Power BI.

| Table | Contenu |
|---|---|
| `daily_consumption_by_site` | consommation quotidienne par site |
| `site_kpis` | KPIs mensuels par site |
| `consumption_by_site_type` | agrégats par type de site |
| `curtailment_opportunities` | opportunités d'effacement et signaux |
| `monthly_consumption_pivot` | synthèse mensuelle |
| `active_alerts` | alertes priorisées |

## Prérequis

1. Ateliers **Bronze** et **Silver** exécutés
2. Lakehouse **`Lakehouse_gold`** créé, ce notebook attaché avec :
   - `Lakehouse_gold` en lakehouse **par défaut** (lecture/écriture)
   - `Lakehouse_silver` et `Lakehouse_bronze` en lakehouses **secondaires** (lecture)

---

## Cellule 1 : Exploration des données Silver disponibles

- **But** : valider que `consumption_enriched` (Silver) est lisible
- **Vérifie** : les **6 sites** du parc via leur `site_id`

In [ ]:
# Lit la table Silver et ne garde que la colonne site_id
# distinct() supprime les doublons : on obtient la liste des sites
df_sites = spark.table("Lakehouse_silver.silver.consumption_enriched").select("site_id").distinct()

print(f"📋 Sites présents dans Silver : {df_sites.count()}")
df_sites.orderBy("site_id").show()

**Attendu**

- **6 sites** présents dans Silver
- **`site_id`** : clé de regroupement de toutes les tables Gold

## Cellule 2 : Vérification des tables source Silver

- **Contrôle d'intégrité** avant de construire Gold
- **`consumption_predictions`** : doit contenir 1 ligne par mesure horaire
- **`sites_reference`** : doit contenir 1 ligne par site

In [ ]:
# count() = nombre de lignes de chaque table source
t1 = spark.table("Lakehouse_silver.silver.consumption_predictions").count()   # prédictions (Silver)
t2 = spark.table("Lakehouse_bronze.bronze.sites_reference").count()           # référentiel des sites (Bronze)

print(f"✅ consumption_predictions : {t1:,} lignes")
print(f"✅ sites_reference         : {t2} lignes")
print("➡️  Prêt pour l'atelier Gold")

**Attendu**

- **4 320 prédictions** = 6 sites × 720 h ✓
- **6 sites** dans le référentiel ✓
- **Intégrité** OK : jointures sûres

## Cellule 3 : Création du schéma gold

- **Schéma `gold`** : regroupe les 6 tables analytiques
- **`IF NOT EXISTS`** : **idempotent**, ré-exécutable sans erreur

In [ ]:
# Crée le schéma "gold" dans le lakehouse par défaut s'il n'existe pas encore
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
print("✅ Schéma gold prêt")

**Attendu**

- **Schéma `gold`** créé dans le lakehouse par défaut (`Lakehouse_gold`)
- **Tables** accessibles sous `gold.<table>`

## Cellule 4 : Import des librairies PySpark

- **`pyspark.sql.functions as F`** : agrégations, conditions, dates
- Import **unique**, utilisé par toutes les cellules suivantes

In [ ]:
# Fonctions Spark (sum, avg, when, col, round...) importées sous l'alias F
from pyspark.sql import functions as F
print("✅ Fonctions PySpark importées")

**Attendu**

- **`F`** disponible : `sum`, `avg`, `max`, `when`, `round`, `to_date`…
- **Pas de sortie de calcul** : seul le message de confirmation s'affiche

## Cellule 5 : Chargement de consumption_enriched

- **Charge** la table Silver enrichie en DataFrame Spark
- **Contenu** : consommation horaire, `baseline_7d_mw`, `anomaly`, `price_eur_mwh`, météo
- **`count()`** : valide le chargement

In [ ]:
# Charge la table Silver enrichie en DataFrame (rien n'est calculé avant le count)
df_enriched = spark.table("Lakehouse_silver.silver.consumption_enriched")
print(f"📊 consumption_enriched : {df_enriched.count():,} lignes | {len(df_enriched.columns)} colonnes")

**Attendu**

- **4 320 lignes** = 6 sites × 720 h
- **27 colonnes** : consommation, baseline, anomalies, prix, météo
- **Cœur factuel** de Gold

## Cellule 6 : Chargement de consumption_predictions

- **Charge** les prédictions et leurs écarts (`prediction_error_mw`)
- **Sert à** : mesurer la fiabilité du modèle et filtrer les faux positifs

In [ ]:
# Charge les prédictions du modèle (une ligne par heure et par site)
df_predictions = spark.table("Lakehouse_silver.silver.consumption_predictions")
print(f"📊 consumption_predictions : {df_predictions.count():,} lignes")

**Attendu**

- **4 320 lignes** : 100 % de couverture, aucune prédiction manquante
- **Sera jointe** à `consumption_enriched` (cellule 7)

## Cellule 7 : Jointure enriched + predictions (dataset Gold source)

- **Jointure** `enriched` + `predictions` sur (`hour`, `site_id`)
- **`left join`** : aucune mesure réelle perdue si une prédiction manque
- **Résultat** : dataset source de toutes les tables Gold

In [ ]:
from pyspark.sql import functions as F

# Jointure sur (hour, site_id) : on ajoute prédiction et erreur à chaque mesure réelle
# left join = on garde toutes les mesures, même sans prédiction
df = df_enriched.join(
    df_predictions.select("hour", "site_id",
                          F.col("prediction").alias("predicted_consumption_mw"),  # renommage pour la clarté
                          "prediction_error_mw"),
    on=["hour", "site_id"],
    how="left"
)
print(f"📊 Dataset Gold source : {df.count():,} lignes (attendu : 4 320)")

# Aperçu des colonnes utiles pour la suite (100 premières lignes)
df.select("hour", "site_id", "avg_consumption_mw",
          "predicted_consumption_mw", "prediction_error_mw", "anomaly", "price_eur_mwh").show(100)

**Attendu**

- **4 320 = 4 320** : jointure complète, aucune perte
- **`anomaly`** : `✅ Normal` ou `⚠️ ANOMALIE`
- **`prediction_error_mw`** : écart réel vs prédit
- **`price_eur_mwh`** : prix spot horaire (≈ 32 à 230 €/MWh sur le jeu de données)
- **Prêt** pour les agrégations

## Cellule 8 : Agrégation journalière par site (df_daily + table gold.daily_consumption_by_site)

- **Agrégation** horaire → **journalière** par site (`df_daily`)
- **Calculs** : somme, moyenne, pic, écart de prédiction, prix, température
- **Jointure** `sites_reference` pour `site_type` et `capacity_mw`
- **Usage** : dashboard opérationnel quotidien

In [ ]:
# --- Étape 1 : ajouter site_type, capacité, région, flexibilité depuis le référentiel ---
df_sites_ref = spark.table("Lakehouse_bronze.bronze.sites_reference")

# Dropper explicitement les colonnes dupliquées de df avant la jointure
cols_to_drop = [c for c in ["site_type", "capacity_mw", "region", "flexible"] if c in df.columns]
df = df.drop(*cols_to_drop)

# Jointure propre
df = df.join(
    df_sites_ref.select("site_id", "site_type", "capacity_mw", "region", "flexible"),
    on="site_id", how="left"
)

# Vérification
print("Colonnes après jointure:", df.columns)

# --- Étape 2 : agréger l'horaire en journalier (une ligne par jour et par site) ---
df_daily = df.groupBy(
    F.to_date("hour").alias("date"),                 # heure -> jour
    "site_id", "site_type", "capacity_mw"
).agg(
    F.sum("avg_consumption_mw").alias("total_consumption_mwh"),   # énergie du jour (somme des 24 h)
    F.avg("avg_consumption_mw").alias("avg_consumption_mw"),      # puissance moyenne
    F.max("max_consumption_mw").alias("peak_consumption_mw"),     # pic de charge
    F.avg("baseline_7d_mw").alias("avg_baseline_mw"),             # référence 7 jours
    F.avg("predicted_consumption_mw").alias("avg_predicted_mw"),  # prédiction moyenne
    F.avg("prediction_error_mw").alias("avg_prediction_error_mw"),
    F.avg("price_eur_mwh").alias("avg_price_eur_mwh"),
    F.max("price_eur_mwh").alias("peak_price_eur_mwh"),
    F.avg("temperature_c").alias("avg_temperature_c"),
    F.sum(F.when(F.col("anomaly") == "⚠️ ANOMALIE", 1).otherwise(0)).alias("nb_anomalies"),  # heures anormales
    F.count("*").alias("nb_measurements")                         # nombre d'heures du jour
).orderBy(F.desc("date"), "site_id")

print(f"✅ df_daily prêt : {df_daily.count()} lignes (attendu : 6 sites × 30 jours = 180)")

**Attendu**

- **180 lignes** = 6 sites × 30 jours
- **Clés** : `total_consumption_mwh` (somme), `peak_consumption_mw` (max), `avg_price_eur_mwh`
- **Dimensions** : `site_type`, `capacity_mw` pour les pivots

## Cellule 9 : Sauvegarde de daily_consumption_by_site

- **Écriture Delta** : `gold.daily_consumption_by_site`
- **Delta** : transactions **ACID**, versioning, *time travel*
- **`overwrite`** : ré-exécution **idempotente**

In [ ]:
# Écrit df_daily en table Delta ; overwrite = remplace la table à chaque exécution
df_daily.write.mode("overwrite").format("delta").saveAsTable("gold.daily_consumption_by_site")
print(f"✅ gold.daily_consumption_by_site : {df_daily.count()} lignes")

# Aperçu : 6 lignes du jour le plus récent
df_daily.show(6)

**Attendu**

- **180 lignes** écrites en Delta
- **Échelle** : ex. 30/01, **SITE_IND_001** ≈ 68,6 MWh vs **SITE_RES_001** ≈ 5,2 MWh (≈ 13 : 1)
- **Ordre de grandeur** cohérent avec les capacités (5 MW vs 0,8 MW)
- **Usage** : série temporelle par site dans Power BI

## Cellule 10 : KPIs mensuels par site (df_kpis)

- **Synthèse mensuelle** par site (`df_kpis`) depuis le daily
- **KPIs** : énergie totale, **`load_factor_pct`**, `anomaly_rate_pct`, `prediction_error_pct`, prix moyen
- **Load factor** : < 30 % sous-utilisé · 50-70 % optimal · > 80 % risque de saturation

In [ ]:
# KPIs mensuels : on regroupe les lignes journalières par site
df_kpis = df_daily.groupBy("site_id", "site_type", "capacity_mw").agg(
    F.sum("total_consumption_mwh").alias("total_consumption_month_mwh"),   # énergie du mois
    F.avg("avg_consumption_mw").alias("avg_consumption_mw"),
    # Facteur de charge = puissance moyenne / capacité × 100
    F.round(F.avg("avg_consumption_mw") / F.col("capacity_mw") * 100, 2).alias("load_factor_pct"),
    # Taux d'anomalies = heures anormales / heures mesurées × 100
    F.round(F.sum("nb_anomalies") * 100.0 / F.sum("nb_measurements"), 2).alias("anomaly_rate_pct"),
    F.round(F.avg("avg_prediction_error_mw"), 3).alias("avg_prediction_error_mw"),
    # Erreur de prédiction relative à la consommation moyenne (%)
    F.round(F.avg("avg_prediction_error_mw") / F.avg("avg_consumption_mw") * 100, 1).alias("prediction_error_pct"),
    F.round(F.avg("avg_price_eur_mwh"), 2).alias("avg_electricity_price_eur_mwh"),
    F.countDistinct("date").alias("nb_days")                               # jours couverts
).orderBy(F.desc("total_consumption_month_mwh"))

# Cette cellule ne calcule que le DataFrame : on affiche donc un résumé lisible
print(f"✅ df_kpis calculé : {df_kpis.count()} lignes (1 par site)")
df_kpis.select("site_id", "total_consumption_month_mwh", "load_factor_pct",
               "anomaly_rate_pct", "prediction_error_pct").show(truncate=False)

**Attendu**

- **1 ligne par site** (6 lignes)
- **Load factor** : calculé sur `capacity_mw`
- **`anomaly_rate_pct`** et **`prediction_error_pct`** : *health check* mensuel

## Cellule 11 : Enrichissement avec région et flexibilité

- **Enrichit** les KPIs avec `region` et `flexible` (depuis `sites_reference`)
- **`flexible`** : le site peut-il participer à l'**effacement** (cellules 14-16) ?

In [ ]:
# Ajoute la région et le flag "flexible" aux KPIs (jointure sur site_id)
df_sites_ref = spark.table("Lakehouse_bronze.bronze.sites_reference")
df_kpis = df_kpis.join(
    df_sites_ref.select("site_id", "region", "flexible"),
    on="site_id", how="left"
)

# Contrôle : colonnes ajoutées et sites éligibles à l'effacement
print("✅ Colonnes ajoutées : region, flexible")
print("🔌 Sites flexibles :", [r["site_id"] for r in df_kpis.filter("flexible = true").select("site_id").orderBy("site_id").collect()])
df_kpis.select("site_id", "region", "flexible").orderBy("site_id").show(truncate=False)

**Attendu**

- **Chaque site** porte maintenant sa **région** et son flag **`flexible`**
- **Questions** possibles : potentiel d'effacement par région, sites contributeurs aux pics

## Cellule 12 : Sauvegarde gold.site_kpis

- **Écriture** de `gold.site_kpis` : 1 ligne par site
- **Table la plus lue** par les décideurs (exploitation, analystes, conformité)

In [ ]:
# Écrit les KPIs en table Delta (overwrite = ré-exécutable)
df_kpis.write.mode("overwrite").format("delta").saveAsTable("gold.site_kpis")
print(f"✅ gold.site_kpis : {df_kpis.count()} lignes")

# Affiche la table complète (6 lignes)
df_kpis.show()

**Attendu**

- **Load factor** : **28 à 51 %** (industriels ≈ 50 %, résidentiels ≈ 28 %)
- **Anomalies** : **0 à 2,2 %** (résidentiels les plus touchés)
- **Erreur de prédiction** : **28 à 47 %** de la consommation moyenne → modèle **peu précis**
- **`flexible`** : IND et RES = `true`, COM = `false`
- **Alimente** : dashboard mensuel, conformité, benchmark inter-sites

## Cellule 13 : Consommation par type de site (gold.consumption_by_site_type)

- **Changement de granularité** : par **type de site** (Industrie / Commercial / Résidentiel) et par jour
- **Calculs** : consommation, capacité totale, **load factor** segment, prix moyen, nombre de sites
- **Usage** : comparer les 3 segments

In [ ]:
# Agrégation par jour et par type de site (Industrie / Commercial / Résidentiel)
df_by_type = df.groupBy(
    F.to_date("hour").alias("date"),
    "site_type"
).agg(
    F.sum("avg_consumption_mw").alias("total_consumption_mwh"),
    F.avg("avg_consumption_mw").alias("avg_consumption_mw"),
    F.sum("capacity_mw").alias("total_capacity_mw"),    # capacité cumulée (ligne horaire × sites)
    # Load factor du segment = consommation / capacité × 100
    F.round(F.sum("avg_consumption_mw") / F.sum("capacity_mw") * 100, 2).alias("load_factor_pct"),
    F.avg("price_eur_mwh").alias("avg_price_eur_mwh"),
    F.countDistinct("site_id").alias("nb_sites")        # sites dans le segment
).orderBy(F.desc("date"), "site_type")

# Sauvegarde Delta
df_by_type.write.mode("overwrite").format("delta").saveAsTable("gold.consumption_by_site_type")
print(f"✅ gold.consumption_by_site_type : {df_by_type.count()} lignes (attendu : 3 types × 30 jours = 90)")

# Aperçu des 3 segments pour le dernier jour
df_by_type.show(3)

**Attendu**

- **90 lignes** = 3 types × 30 jours
- **Colonnes** : consommation, `total_capacity_mw`, `load_factor_pct`, prix moyen, `nb_sites` (= 2)
- **Contraste** : ex. 30/01 : Industrie ≈ 58 % de load factor vs Résidentiel ≈ 28 %
- **Usage** : 3 courbes superposées dans Power BI

## Cellule 14 : Filtrage des sites flexibles éligibles à l'effacement

- **Effacement (curtailment)** : réduction volontaire et temporaire de la consommation
- **Filtre 1** : `flexible == True`
- **Filtre 2** : `avg_consumption_mw > 0.5` (évite le bruit)
- **Sortie** : `df_flex`, entrée des cellules 15-16

In [ ]:
# Ne garde que les sites flexibles ET une consommation > 0,5 MW (évite le bruit)
df_flex = df.filter(
    (F.col("flexible") == True) &
    (F.col("avg_consumption_mw") > 0.5)
)
print(f"📊 Lignes éligibles effacement : {df_flex.count():,}")

# Contrôle : quels sites sont retenus et combien de lignes chacun
df_flex.groupBy("site_id").count().orderBy("site_id").show()

**Attendu**

- **1 479 lignes** éligibles
- **4 sites** `flexible` (IND + RES) et consommation > 0,5 MW
- **COM** exclus (non flexibles)

## Cellule 15 : Calcul du potentiel d'effacement (curtailment)

- **`ratio_vs_baseline`** : consommation / baseline 7 j (> 1 = surcharge)
- **`curtailment_potential_mw`** : **30 %** de la consommation (limite opérationnelle)
- **`potential_gain_eur`** : MW effacés × (prix spot − prix contractuel), sinon 0
- **`action_signal`** : 🔴 > 300 · 🟠 > 200 · 🟢 < 50 · ⚪ sinon (€/MWh)

In [ ]:
# Calcul des indicateurs d'effacement sur chaque ligne éligible
df_curtailment = df_flex.withColumn(
    "ratio_vs_baseline",                                      # >1 = consomme plus que d'habitude
    F.round(F.col("avg_consumption_mw") / F.col("baseline_7d_mw"), 2)
).withColumn(
    "curtailment_potential_mw",                               # puissance effaçable
    F.round(F.col("avg_consumption_mw") * 0.3, 3)         # 30% effacement
).withColumn(
    "potential_gain_eur",                                     # gain seulement si prix spot > seuil contractuel
    F.when(
        F.col("price_eur_mwh") > F.col("curtailment_price_eur_mwh"),
        F.round((F.col("avg_consumption_mw") * 0.3) *
                (F.col("price_eur_mwh") - F.col("curtailment_price_eur_mwh")), 2)
    ).otherwise(0)
).withColumn(
    "action_signal",                                          # recommandation selon le prix spot
    F.when(F.col("price_eur_mwh") > 300, "🔴 EFFACEMENT MAX")
     .when(F.col("price_eur_mwh") > 200, "🟠 EFFACEMENT PARTIEL")
     .when(F.col("price_eur_mwh") < 50,  "🟢 CONSOMMER")
     .otherwise("⚪ Surveiller")
)

# Cette cellule ne fait que calculer : on affiche donc un résumé
print(f"✅ Colonnes ajoutées : ratio_vs_baseline, curtailment_potential_mw, potential_gain_eur, action_signal")
print(f"📊 Heures évaluées : {df_curtailment.count():,}")
print("📊 Répartition des signaux d'action :")
df_curtailment.groupBy("action_signal").count().orderBy(F.desc("count")).show(truncate=False)
gain = df_curtailment.agg(F.round(F.sum("potential_gain_eur"), 0)).collect()[0][0]
print(f"💶 Gain potentiel total : {gain:,.0f} €")

**Attendu**

- **Nouvelles colonnes** : `ratio_vs_baseline`, `curtailment_potential_mw`, `potential_gain_eur`, `action_signal`
- **Exemple** : 3,3 MW × 30 % = **0,99 MW** ; prix 230 € − seuil 102 € → gain ≈ **127 €**
- **Seuils** : prix max observé ≈ **230 €/MWh** → **aucun 🔴** (> 300)

## Cellule 16 : Sauvegarde gold.curtailment_opportunities

- **Écriture** de `gold.curtailment_opportunities`
- **Tri** par `potential_gain_eur` décroissant : meilleures heures en premier
- **Usage** : alertes opérationnelles, optimisation des contrats de flexibilité

In [ ]:
# Sélectionne et ordonne les colonnes utiles, triées par gain décroissant
df_curtailment = df_curtailment.select(
    "hour", "site_id", "site_type", "curtailment_price_eur_mwh",
    "avg_consumption_mw", "baseline_7d_mw", "ratio_vs_baseline",
    "price_eur_mwh", "curtailment_potential_mw", "potential_gain_eur", "action_signal"
).orderBy(F.desc("potential_gain_eur"))

# Sauvegarde Delta
df_curtailment.write.mode("overwrite").format("delta").saveAsTable("gold.curtailment_opportunities")
print(f"✅ gold.curtailment_opportunities : {df_curtailment.count():,} lignes")

# Résumé : heures rentables (gain > 0) et meilleure opportunité
nb_gain = df_curtailment.filter("potential_gain_eur > 0").count()
best = df_curtailment.first()
print(f"💶 Heures avec gain > 0 : {nb_gain:,} sur {df_curtailment.count():,}")
print(f"🏆 Meilleure opportunité : {best['site_id']} le {best['hour']} → {best['potential_gain_eur']} €")

# Aperçu des 50 meilleures heures
df_curtailment.filter("potential_gain_eur > 0").show(50)

**Attendu**

- **1 479 lignes** écrites
- **Répartition** : ⚪ Surveiller 1 255 · 🟢 Consommer 216 · 🟠 Partiel 8 · 🔴 Max 0
- **Gain** : max ≈ **127 €/heure** (IND_001, 24/01 à 13 h), total potentiel ≈ **16 100 €**
- **496 heures** avec gain > 0 ; les autres : prix < seuil contractuel (gain = 0)

## Cellule 17 : Pivot mensuel (gold.monthly_consumption_pivot)

- **Pivot mensuel** : 1 ligne par site (`year`, `month`)
- **Colonnes** : énergie totale, load factor moyen, prix moyen, anomalies
- **Plus léger** que `site_kpis`, adapté aux **visuels croisés** Power BI

In [ ]:
# Synthèse mensuelle : une ligne par site, par année et par mois
df_monthly = df_daily.groupBy(
    "site_id", "site_type",
    F.year("date").alias("year"),
    F.month("date").alias("month")
).agg(
    F.sum("total_consumption_mwh").alias("total_mwh"),                    # énergie du mois
    F.avg(F.col("avg_consumption_mw") / F.col("capacity_mw") * 100).alias("avg_load_factor_pct"),
    F.avg("avg_price_eur_mwh").alias("avg_price_eur_mwh"),
    F.sum("nb_anomalies").alias("total_anomalies")                        # heures anormales du mois
).orderBy("site_id")

# Sauvegarde Delta
df_monthly.write.mode("overwrite").format("delta").saveAsTable("gold.monthly_consumption_pivot")
print(f"✅ gold.monthly_consumption_pivot : {df_monthly.count()} lignes")
df_monthly.show()

**Attendu**

- **6 lignes** : 1 par site, janvier 2025
- **Énergie** : IND_001 ≈ 1 801 MWh, RES_002 ≈ 122 MWh
- **Prix moyen** ≈ 95,8 €/MWh
- **Anomalies** : **RES_002 = 16**, RES_001 = 14, IND_002 = 4, COM_002 = 2, COM_001 = 0
- **Multi-mois** : révélera la saisonnalité

## Cellule 18 : Détection des événements d'alerte

- **3 critères** d'alerte, combinés par **OU** :
  - **Anomalie** détectée en Silver (z-score)
  - **Prix spot** > 300 €/MWh
  - **Erreur de prédiction** > 1 MW
- **Sortie** : `df_alerts`

In [ ]:
# Garde les heures qui remplissent AU MOINS UN des 3 critères (OU logique "|")
df_alerts = df.filter(
    (F.col("anomaly") == "⚠️ ANOMALIE") |          # 1. anomalie détectée en Silver
    (F.col("price_eur_mwh") > 300)        |          # 2. prix spot extrême
    (F.col("prediction_error_mw") > 1.0)             # 3. modèle de prédiction en défaut
)
print(f"📊 Événements d'alerte : {df_alerts.count():,}")

# Détail par critère (une heure peut compter dans plusieurs)
print(f"   • anomalies            : {df_alerts.filter(F.col('anomaly') == '⚠️ ANOMALIE').count():,}")
print(f"   • prix > 300 €/MWh     : {df_alerts.filter('price_eur_mwh > 300').count():,}")
print(f"   • erreur prédiction > 1 MW : {df_alerts.filter('prediction_error_mw > 1.0').count():,}")

**Attendu**

- **260 alertes** sur 4 320 mesures (≈ 6 %)
- **36** anomalies
- **0** prix > 300 €/MWh (max ≈ 230)
- **231** erreurs de prédiction > 1 MW (dont 7 aussi en anomalie)
- **Total** : 36 + 231 − 7 = **260**

## Cellule 19 : Qualification et priorisation des alertes

- **`alert_type`** : oriente vers la bonne équipe (technique, trading, data science)
- **`priority`** :
  - **CRITIQUE** : anomalie **et** prix > 300
  - **HAUTE** : anomalie **ou** prix > 300
  - **MOYENNE** : erreur de prédiction seule

In [ ]:
# alert_type : le 1er critère vrai l'emporte (ordre des when)
# priority   : CRITIQUE (anomalie + prix) > HAUTE (l'un des deux) > MOYENNE (erreur de prédiction seule)
df_alerts = df_alerts.withColumn(
    "alert_type",
    F.when(F.col("anomaly") == "⚠️ ANOMALIE", "Consommation anormale")
     .when(F.col("price_eur_mwh") > 300,        "Prix spot très élevé")
     .when(F.col("prediction_error_mw") > 1.0,  "Erreur prédiction élevée")
     .otherwise("Autre")
).withColumn(
    "priority",
    F.when((F.col("anomaly") == "⚠️ ANOMALIE") & (F.col("price_eur_mwh") > 300), "CRITIQUE")
     .when((F.col("anomaly") == "⚠️ ANOMALIE") | (F.col("price_eur_mwh") > 300), "HAUTE")
     .when(F.col("prediction_error_mw") > 1.0, "MOYENNE")
     .otherwise("BASSE")
)

# Cette cellule ne fait que calculer : on affiche la répartition type × priorité
print("✅ Colonnes ajoutées : alert_type, priority")
df_alerts.groupBy("alert_type", "priority").count().orderBy("priority", "alert_type").show(truncate=False)

**Attendu**

- **Types** : *Consommation anormale* (36) · *Erreur prédiction élevée* (224)
- **Priorités** : **HAUTE** 36 · **MOYENNE** 224 · **CRITIQUE** 0
- **CRITIQUE** : exige un prix > 300 €/MWh, jamais atteint ici

## Cellule 20 : Sauvegarde gold.active_alerts

- **Écriture** de `gold.active_alerts`, triée par priorité puis par date
- **Usage** : notifications (mail, SMS) et dashboard temps réel

In [ ]:
# Colonnes finales (alert_time = heure de l'alerte) puis tri :
# priorité (CRITIQUE d'abord) puis alerte la plus récente
df_alerts = df_alerts.select(
    F.col("hour").alias("alert_time"),
    "site_id", "site_type", "alert_type",
    "avg_consumption_mw", "baseline_7d_mw", "predicted_consumption_mw",
    "price_eur_mwh", "priority"
).orderBy(
    F.when(F.col("priority") == "CRITIQUE", 1)
     .when(F.col("priority") == "HAUTE",    2)
     .when(F.col("priority") == "MOYENNE",  3)
     .otherwise(4),
    F.desc("alert_time")
)

# Sauvegarde Delta
df_alerts.write.mode("overwrite").format("delta").saveAsTable("gold.active_alerts")
print(f"✅ gold.active_alerts : {df_alerts.count():,} lignes")

# Aperçu : 5 premières alertes CRITIQUE/HAUTE (s'il y en a)
df_alerts.filter("priority IN ('CRITIQUE', 'HAUTE')").show(5)

**Attendu**

- **260 lignes** écrites
- **Haut du tableau** : alertes **HAUTE** (anomalies) les plus récentes
- **Ex.** : SITE_RES_002, 30/01 à 04 h : 0,50 MW vs baseline 0,16 MW (≈ 3× la normale)
- **Action** : vérifier le site ; les alertes MOYENNE relèvent du modèle

## Cellule 21 : Récapitulatif final des tables Gold

- **Contrôle final** : les 6 tables Gold existent et sont lisibles
- **Prêtes** pour le SQL Endpoint et Power BI

In [ ]:
# Liste des 6 tables Gold à contrôler
tables_gold = [
    "gold.daily_consumption_by_site",
    "gold.site_kpis",
    "gold.consumption_by_site_type",
    "gold.curtailment_opportunities",
    "gold.monthly_consumption_pivot",
    "gold.active_alerts"
]

print("=" * 55)
print("📦 TABLES GOLD — RÉCAPITULATIF FINAL")
print("=" * 55)
for table in tables_gold:
    # Relit chaque table depuis le lakehouse pour confirmer qu'elle existe
    count = spark.table(f"Lakehouse_gold.{table}").count()
    print(f"  ✅ {table:<40} {count:>7,} lignes")
print("=" * 55)
print("➡️  Prochaine étape : NB_06 Visualisation matplotlib")

**Attendu**

- **6 tables Gold** livrées :

| Table | Lignes | Granularité |
|---|---|---|
| **daily_consumption_by_site** | 180 | jour × site |
| **site_kpis** | 6 | site |
| **consumption_by_site_type** | 90 | jour × type |
| **curtailment_opportunities** | 1 479 | heure × site flexible |
| **monthly_consumption_pivot** | 6 | mois × site |
| **active_alerts** | 260 | événement |

- **Prochaine étape** : visualisation (NB_06) puis Power BI